# RecursiveMAS Latent Visualizer — Day 1 & 2 Setup

**Runtime richiesto**: A100 40GB  
Menu → Runtime → Cambia tipo di runtime → A100

Questo notebook copre:
- Giorno 1: dipendenze, forward pass di verifica
- Giorno 2: caricamento Sequential-Light completo + primo loop ricorsivo

## 0. Verifica GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU non disponibile — cambia runtime!"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.0f} GB")

## 1. Installa dipendenze

In [ ]:
!pip install transformers torch gradio scipy matplotlib huggingface_hub accelerate spaces -q
print("Dipendenze installate ✓")

## 2. Clona il repo e imposta il path

In [ ]:
import os, sys

REPO_URL = "https://github.com/YOUR_USERNAME/LatentScope.git"  # <-- aggiorna con il tuo URL
REPO_DIR = "/content/LatentScope"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo pronto in {REPO_DIR} ✓")

## 3. Autenticazione HuggingFace

In [ ]:
from huggingface_hub import login
# Incolla il tuo token HF (Read) — non committarlo mai nel repo
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

## GIORNO 1 — Forward pass di verifica

Carica solo il Planner base (Qwen3-1.7B) per verificare che GPU e token HF funzionino.
Checkpoint: `Shape logits: [1, seq_len, vocab_size]` senza errori CUDA.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id  = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)

inputs = tokenizer("Quanto fa 15 × 23?", return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

print("Shape logits:      ", outputs.logits.shape)
print("Shape hidden[-1]:  ", outputs.hidden_states[-1].shape)
print("Giorno 1 OK ✓")

del model  # libera VRAM prima di caricare il sistema completo
torch.cuda.empty_cache()

## GIORNO 2 — Caricamento Sequential-Light completo

Carica Planner + Critic + Solver + 3 OuterLinks (math).  
Checkpoint: VRAM usata < 12 GB, nessun OOM.

In [ ]:
from src.models.load_sequential import load_sequential_light

mas = load_sequential_light(task="math")
print("\nSequential-Light caricato ✓")
print("Chiavi disponibili:", [k for k in mas.keys() if 'tokenizer' not in k])

## GIORNO 2 — Primo loop ricorsivo

Checkpoint:
- `answer` non vuoto
- 3 hidden states con shape `[1536]`
- Similarity tra round < 0.9999

In [ ]:
from src.inference.sequential_loop import run_recursive_loop
from src.inference.hidden_states import extract_hidden_states, verify_hidden_states_differ
from src.inference.metrics import compute_round_metrics

question = "Quanto fa 15 × 23?"
results  = run_recursive_loop(question, mas)
hidden   = extract_hidden_states(results)

print("Risposta finale:", results["answer"])
print()
print("Shape hidden states:", [h.shape for h in hidden])
print()
print("Verifica hidden states:")
verify_hidden_states_differ(hidden)

print()
print("Metriche per round:")
metrics = compute_round_metrics(hidden, results["logits"])
for i, m in enumerate(metrics):
    sim = f"{m['cosine_sim']:.4f}" if m['cosine_sim'] is not None else "N/A"
    print(f"  Round {i+1}: cosine_sim={sim}  entropy={m['entropy']:.3f}  confidence={m['confidence']:.4f}")

## Checkpoint finale Giorno 2

Se il blocco precedente stampa 3 righe senza errori:
- aggiorna `STATUS.md` → Giorno 2: COMPLETATO
- prossimo step: `phase2-core.md` Giorno 3 — metriche e UI Feature 1